In [1]:
import connectorx as cx
import evadb
import ffnn
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
from torch.utils.data import DataLoader
import utils
import load_data_to_db
import collections
import os
import h5py
from abc import ABC, abstractmethod
from models.preprocessing.inputs import SparseFeat, DenseFeat, VarLenSparseFeat
from models.dssm import DSSM_Torch, DSSM_TF, get_var_feature, get_test_var_feature
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
# from category_encoders.ordinal import OrdinalEncoder
from tqdm.auto import tqdm
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, pandas_udf, when
import pyspark.sql.functions as F
from pyspark.sql.types import ArrayType, FloatType, StringType, IntegerType
from dssm_evadb import DSSM_Moel_Wrapper
import pickle
import multiprocessing as mp
from pipeline import Pipeline
from sklearn.model_selection import train_test_split
import pyarrow.parquet as pq


2025-01-07 07:53:53.672802: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-01-07 07:53:53.672829: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.1.1 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
2025-01-07 07:53:57.010952: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or d

In [5]:
# Use case 8, trainig query
query_to_fetch_training_data = """
SELECT 
        o_order_id,
        department,
        quantity,
        SUM(quantity) AS scan_count,                -- Equivalent to np.sum(x)
        MIN(EXTRACT(DOW FROM date)) AS weekday,     -- Equivalent to np.min(x) for weekday
        MIN(trip_type) AS trip_type                 -- Equivalent to np.min(x) for trip_type
    FROM tpcxai_order_training 
    JOIN tpcxai_lineitem_training ON o_order_id = li_order_id 
    JOIN tpcxai_product_training ON li_product_id = p_product_id
    GROUP BY o_order_id, date, department, quantity
"""

In [6]:
df = utils.fetch_data_from_postgres_via_connectorx(query_to_fetch_training_data)

In [21]:
le_department = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
le_department.fit(df[['department']])
df['department_encoded'] = le_department.transform(df[['department']])
le_trip_type = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
le_trip_type.fit(df[['trip_type']])
df['trip_type_encoded'] = le_trip_type.transform(df[['trip_type']])

In [23]:
df.head()

,o_order_id,department,quantity,scan_count,weekday,trip_type,department_encoded,trip_type_encoded
0,1,PRODUCE,2,2.0,3.0,8,40.0,5.0
1,2,FINANCIAL SERVICES,1,1.0,2.0,3,13.0,0.0
2,2,FINANCIAL SERVICES,2,2.0,2.0,3,13.0,0.0
3,2,FINANCIAL SERVICES,3,9.0,2.0,3,13.0,0.0
4,2,IMPULSE MERCHANDISE,2,4.0,2.0,3,23.0,0.0


In [30]:
df['scan_count'] = df['scan_count'].astype(int)
df['weekday'] = df['weekday'].astype(int)
X_features = df[['quantity', 'scan_count', 'weekday', 'department_encoded']].values.astype(float)
y = df['trip_type_encoded'].values.astype(float)
num_y = len(np.unique(y))

In [31]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(256, activation='relu', input_shape=(4,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(num_y, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
X_train, X_test, y_train, y_test = train_test_split(X_features, y, test_size=0.2, random_state=0)

In [ ]:
model.fit(X_train, y_train, epochs=10, batch_size=2048*2, validation_data=(X_test, y_test))

In [34]:
model.save('../../resources/model/tpcxai_sf1/final/tf/usecase8.h5', include_optimizer=False)
model_weights = model.get_weights()
with h5py.File('../../resources/model/tpcxai_sf1/final/velox/usecase8_ffnn_weight.h5', 'w') as f:
    f.create_dataset('w1', data=model_weights[0])
    f.create_dataset('b1', data=model_weights[1])
    f.create_dataset('w2', data=model_weights[2])
    f.create_dataset('b2', data=model_weights[3])
    f.create_dataset('w3', data=model_weights[4])
    f.create_dataset('b3', data=model_weights[5])
    f.create_dataset('w4', data=model_weights[6])
    f.create_dataset('b4', data=model_weights[7])

In [35]:
with open('../../resources/model/tpcxai_sf1/final/tf/usecase8_le_dept.pkl', 'wb') as f:
    pickle.dump(le_department, f)
with open('../../resources/model/tpcxai_sf1/final/tf/usecase8_le_trip_type.pkl', 'wb') as f:
    pickle.dump(le_trip_type, f)

In [36]:
# Use case 8, serving query
query_to_fetch_serving_data = """
SELECT 
        o_order_id,
        department,
        quantity,
        SUM(quantity) AS scan_count,                -- Equivalent to np.sum(x)
        MIN(EXTRACT(DOW FROM date)) AS weekday     -- Equivalent to np.min(x) for weekday
    FROM tpcxai_order_serving 
    JOIN tpcxai_lineitem_serving ON o_order_id = li_order_id 
    JOIN tpcxai_product_serving ON li_product_id = p_product_id
    GROUP BY o_order_id, date, department, quantity
"""

In [37]:
df_serve = utils.fetch_data_from_postgres_via_psycopg2(query_to_fetch_serving_data)

In [38]:
df_serve['department_encoded'] = le_department.transform(df_serve[['department']])
df_serve['scan_count'] = df_serve['scan_count'].astype(int)
df_serve['weekday'] = df_serve['weekday'].astype(int)

In [39]:
X_serve = df_serve[['quantity', 'scan_count', 'weekday', 'department_encoded']].values.astype(float)
y_pred = model.predict(X_serve)